# Cillian AI Studio
10 Open-Source Modelle gratis auf Google Colab T4 GPU.

| # | Modell | Typ | ca. Dauer |
|---|--------|-----|-----------|
| 1 | LTX 2.3 | Text→Video | ~2 min |
| 2 | LTX 2.3 | **Image→Video** | ~2 min |
| 3 | Wan 2.1 | Text→Video | ~4 min |
| 4 | Wan 2.1 | **Image→Video** | ~4 min |
| 5 | Flux Dev | Text→Bild (Qualitaet) | ~80s |
| 6 | Flux Schnell | Text→Bild (schnell) | ~15s |
| 7 | SDXL Lightning | Text→Bild | ~30s |
| 8 | PixArt-Sigma | Text→Bild (klein) | ~20s |
| 9 | SD 3.5 Medium | Text→Bild | ~40s |
| 10 | Whisper Large V3 | Sprache→Text | ~5s |
| 11 | Qwen2-Audio | Audio verstehen | ~10s |
| 12 | Kokoro TTS | Text→Sprache | ~2s |

Alle Zellen der Reihe nach ausfuehren. Am Ende bekommst du eine API-URL.

**API Key:** `ironclaw-video-2026`

## 1. GPU Check

In [ ]:
!nvidia-smi
import torch
print(f'\nCUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
assert torch.cuda.is_available(), 'KEINE GPU! Gehe zu Laufzeit > Laufzeittyp aendern > T4 GPU'

## 2. ComfyUI + Nodes installieren (~2 min)

In [ ]:
import os, subprocess

if not os.path.exists('/content/ComfyUI'):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -r requirements.txt -q
else:
    %cd /content/ComfyUI
    print('ComfyUI already installed')

# Custom Nodes
nodes = {
    'ComfyUI-GGUF': 'https://github.com/city96/ComfyUI-GGUF.git',
    'ComfyUI-LTXVideo': 'https://github.com/Lightricks/ComfyUI-LTXVideo.git',
    'ComfyUI-KJNodes': 'https://github.com/kijai/ComfyUI-KJNodes.git',
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
}
for name, url in nodes.items():
    d = f'/content/ComfyUI/custom_nodes/{name}'
    if not os.path.exists(d):
        !git clone --depth 1 {url} {d}
        req = os.path.join(d, 'requirements.txt')
        if os.path.exists(req):
            !pip install -r {req} -q 2>/dev/null

!pip install gguf huggingface_hub hf_transfer -q

# T4 patch: float16 support (T4 has no bfloat16)
for pyfile in ['comfy/model_management.py', 'comfy/sd.py']:
    fp = f'/content/ComfyUI/{pyfile}'
    if os.path.exists(fp):
        with open(fp, 'r') as f: c = f.read()
        if 'working_dtypes = [torch.bfloat16]' in c:
            c = c.replace('working_dtypes = [torch.bfloat16]', 'working_dtypes = [torch.bfloat16, torch.float16]')
            with open(fp, 'w') as f: f.write(c)
            print(f'T4 patch: {pyfile}')

print('\nSetup complete!')

## 3. Google Drive + Download Helper

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CACHE = '/content/drive/MyDrive/ai-models'
os.makedirs(CACHE, exist_ok=True)

DIRS = {}
for d in ['unet', 'clip', 'vae', 'loras', 'checkpoints']:
    p = f'/content/ComfyUI/models/{d}'
    os.makedirs(p, exist_ok=True)
    DIRS[d] = p

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download, snapshot_download
import shutil

def dl(repo, filename, target_dir):
    safe = filename.replace('/', '__')
    cache = os.path.join(CACHE, safe)
    target = os.path.join(target_dir, os.path.basename(filename))
    if os.path.exists(target):
        print(f'  OK: {os.path.basename(filename)}')
        return
    if os.path.exists(cache):
        os.symlink(cache, target)
        print(f'  Cache: {os.path.basename(filename)}')
        return
    print(f'  Downloading: {os.path.basename(filename)}...')
    p = hf_hub_download(repo_id=repo, filename=filename, local_dir='/tmp/hf_dl')
    shutil.move(p, cache)
    os.symlink(cache, target)
    print(f'  Done: {os.path.basename(filename)}')

print('Ready!')

## 4. Video-Modelle herunterladen

In [ ]:
print('=== LTX 2.3 (T2V + I2V, ~13 GB) ===')
dl('unsloth/LTX-2.3-GGUF', 'distilled/ltx-2.3-22b-distilled-Q4_K_S.gguf', DIRS['unet'])
dl('Kijai/LTX2.3_comfy', 'vae/LTX23_video_vae_bf16.safetensors', DIRS['vae'])
dl('Kijai/LTX2.3_comfy', 'vae/LTX23_audio_vae_bf16.safetensors', DIRS['vae'])
dl('Kijai/LTX2.3_comfy', 'loras/ltx-2.3-22b-distilled-lora-dynamic_fro09_avg_rank_105_bf16.safetensors', DIRS['loras'])

print('\n=== Wan 2.1 T2V (~10 GB) ===')
dl('city96/Wan2.1-T2V-14B-gguf', 'wan2.1-t2v-14b-Q4_K_S.gguf', DIRS['unet'])

print('\n=== Wan 2.1 I2V (~10 GB) ===')
dl('city96/Wan2.1-I2V-14B-480p-gguf', 'wan2.1-i2v-14b-480p-Q4_K_S.gguf', DIRS['unet'])

print('\n=== Shared Video Components ===')
dl('Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/vae/wan_2.1_vae.safetensors', DIRS['vae'])
dl('Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', DIRS['clip'])
# CLIP Vision for I2V
dl('Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/clip_vision/clip_vision_h.safetensors', DIRS['clip'])

print('\nVideo models ready!')

## 5. Bild-Modelle herunterladen

In [ ]:
print('=== Flux Dev GGUF (~7 GB) ===')
dl('city96/FLUX.1-dev-gguf', 'flux1-dev-Q4_K_S.gguf', DIRS['unet'])

print('\n=== Flux Schnell GGUF (~7 GB) ===')
dl('city96/FLUX.1-schnell-gguf', 'flux1-schnell-Q4_K_S.gguf', DIRS['unet'])

print('\n=== Shared Flux Components ===')
dl('comfyanonymous/flux_text_encoders', 'clip_l.safetensors', DIRS['clip'])
dl('comfyanonymous/flux_text_encoders', 't5xxl_fp8_e4m3fn.safetensors', DIRS['clip'])
dl('ffxvs/vae-flux', 'ae.safetensors', DIRS['vae'])

print('\n=== SDXL Lightning (~7 GB) ===')
dl('ByteDance/SDXL-Lightning', 'sdxl_lightning_4step_unet.safetensors', DIRS['unet'])

print('\n=== SD 3.5 Medium GGUF (~5 GB) ===')
dl('city96/stable-diffusion-3.5-medium-gguf', 'sd3.5_medium-Q4_K_S.gguf', DIRS['unet'])
dl('Comfy-Org/stable-diffusion-3.5-fp8', 'text_encoders/clip_g.safetensors', DIRS['clip'])
dl('Comfy-Org/stable-diffusion-3.5-fp8', 'text_encoders/clip_l.safetensors', DIRS['clip'])

print('\nImage models ready!')

## 6. Audio-Modelle herunterladen

In [ ]:
!pip install transformers accelerate librosa soundfile kokoro -q

print('=== Whisper Large V3 ===')
snapshot_download('openai/whisper-large-v3', cache_dir=CACHE, ignore_patterns=['*.md', '*.txt'])
print('OK')

print('\n=== Qwen2-Audio ===')
snapshot_download('Qwen/Qwen2-Audio-7B-Instruct', cache_dir=CACHE, ignore_patterns=['*.md', '*.txt'])
print('OK')

print('\n=== Kokoro TTS (82M) ===')
snapshot_download('hexgrad/Kokoro-82M', cache_dir=CACHE, ignore_patterns=['*.md'])
print('OK')

print('\nAudio models ready!')

## 7. ComfyUI starten

In [ ]:
import subprocess, threading, time

comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188',
     '--lowvram', '--force-fp16', '--disable-smart-memory'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd='/content/ComfyUI'
)

for line in iter(comfy_proc.stdout.readline, b''):
    l = line.decode('utf-8', errors='ignore').strip()
    if l: print(l)
    if 'To see the GUI' in l or 'Starting server' in l:
        print('\n=== ComfyUI ready! ===')
        break

## 8. API Server schreiben

In [ ]:
%%writefile /content/api_server.py
"""Cillian AI Studio — API Server"""
import asyncio, json, uuid, os, time, io, tempfile
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.responses import FileResponse, StreamingResponse
import httpx, uvicorn, torch

app = FastAPI(title='Cillian AI Studio')
COMFY = 'http://127.0.0.1:8188'
OUTPUT = '/content/ComfyUI/output'
INPUT = '/content/ComfyUI/input'
CACHE = '/content/drive/MyDrive/ai-models'
KEY = 'ironclaw-video-2026'
os.makedirs(INPUT, exist_ok=True)
os.makedirs(OUTPUT, exist_ok=True)

def ck(k):
    if k != KEY: raise HTTPException(401, 'Bad API key')

# ════════════════════════════════════════════
# Audio Model Manager
# ════════════════════════════════════════════
audio_loaded = None
audio_models = {}

def unload_audio():
    global audio_loaded, audio_models
    if audio_loaded:
        del audio_models[audio_loaded]
        audio_models = {}
        audio_loaded = None
        torch.cuda.empty_cache()

def load_whisper():
    global audio_loaded, audio_models
    if audio_loaded == 'whisper': return audio_models['whisper']
    unload_audio()
    from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        'openai/whisper-large-v3', torch_dtype=torch.float16,
        device_map='auto', cache_dir=CACHE
    )
    processor = AutoProcessor.from_pretrained('openai/whisper-large-v3', cache_dir=CACHE)
    pipe = pipeline('automatic-speech-recognition', model=model, tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor, torch_dtype=torch.float16, device_map='auto')
    audio_models['whisper'] = pipe
    audio_loaded = 'whisper'
    return pipe

def load_qwen():
    global audio_loaded, audio_models
    if audio_loaded == 'qwen': return audio_models['qwen']
    unload_audio()
    from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
    proc = AutoProcessor.from_pretrained('Qwen/Qwen2-Audio-7B-Instruct', cache_dir=CACHE)
    model = Qwen2AudioForConditionalGeneration.from_pretrained(
        'Qwen/Qwen2-Audio-7B-Instruct', torch_dtype=torch.float16,
        device_map='auto', cache_dir=CACHE
    )
    audio_models['qwen'] = (proc, model)
    audio_loaded = 'qwen'
    return proc, model

def load_kokoro():
    global audio_loaded, audio_models
    if audio_loaded == 'kokoro': return audio_models['kokoro']
    unload_audio()
    try:
        from kokoro import KPipeline
        pipe = KPipeline(lang_code='a')
        audio_models['kokoro'] = pipe
        audio_loaded = 'kokoro'
        return pipe
    except Exception as e:
        print(f'Kokoro load error: {e}')
        return None

# ════════════════════════════════════════════
# ComfyUI Workflows
# ════════════════════════════════════════════
def ltx_t2v(prompt, w=768, h=512, steps=8, frames=97):
    return {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'ltx-2.3-22b-distilled-Q4_K_S.gguf'}},
        '2': {'class_type': 'EmptyLTXVLatentVideo', 'inputs': {'width': w, 'height': h, 'length': frames, 'batch_size': 1}},
        '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['10', 0]}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['10', 0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['3',0], 'negative': ['4',0], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': 1.0, 'sampler_name': 'euler', 'scheduler': 'normal', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['9',0]}},
        '8': {'class_type': 'VHS_VideoCombine', 'inputs': {'images': ['6',0], 'frame_rate': 24, 'filename_prefix': 'ltx_t2v', 'format': 'video/h264-mp4', 'save_output': True, 'loop_count': 0, 'pingpong': False}},
        '9': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'LTX23_video_vae_bf16.safetensors'}},
        '10': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'type': 'ltxv'}}
    }

def ltx_i2v(prompt, image_name, w=768, h=512, steps=8, frames=97):
    return {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'ltx-2.3-22b-distilled-Q4_K_S.gguf'}},
        '2': {'class_type': 'LoadImage', 'inputs': {'image': image_name}},
        '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['10', 0]}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['10', 0]}},
        '5': {'class_type': 'LTXVImgToVideo', 'inputs': {'model': ['1',0], 'positive': ['3',0], 'negative': ['4',0], 'image': ['2',0], 'vae': ['9',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': 1.0, 'width': w, 'height': h, 'length': frames}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['9',0]}},
        '8': {'class_type': 'VHS_VideoCombine', 'inputs': {'images': ['6',0], 'frame_rate': 24, 'filename_prefix': 'ltx_i2v', 'format': 'video/h264-mp4', 'save_output': True, 'loop_count': 0, 'pingpong': False}},
        '9': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'LTX23_video_vae_bf16.safetensors'}},
        '10': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'type': 'ltxv'}}
    }

def wan_t2v(prompt, w=832, h=480, steps=20, frames=81):
    return {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'wan2.1-t2v-14b-Q4_K_S.gguf'}},
        '2': {'class_type': 'EmptySD3LatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['10',0]}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['10',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['3',0], 'negative': ['4',0], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': 5.0, 'sampler_name': 'euler', 'scheduler': 'normal', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['9',0]}},
        '8': {'class_type': 'VHS_VideoCombine', 'inputs': {'images': ['6',0], 'frame_rate': 16, 'filename_prefix': 'wan_t2v', 'format': 'video/h264-mp4', 'save_output': True, 'loop_count': 0, 'pingpong': False}},
        '9': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'wan_2.1_vae.safetensors'}},
        '10': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'type': 'wan'}}
    }

def wan_i2v(prompt, image_name, w=832, h=480, steps=20, frames=81):
    return {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'wan2.1-i2v-14b-480p-Q4_K_S.gguf'}},
        '2': {'class_type': 'LoadImage', 'inputs': {'image': image_name}},
        '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['10',0]}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['10',0]}},
        '11': {'class_type': 'CLIPVisionLoader', 'inputs': {'clip_name': 'clip_vision_h.safetensors'}},
        '12': {'class_type': 'CLIPVisionEncode', 'inputs': {'clip_vision': ['11',0], 'image': ['2',0]}},
        '13': {'class_type': 'unCLIPConditioning', 'inputs': {'conditioning': ['3',0], 'clip_vision_output': ['12',0], 'strength': 1.0, 'noise_augmentation': 0.0}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['13',0], 'negative': ['4',0], 'latent_image': ['14',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': 5.0, 'sampler_name': 'euler', 'scheduler': 'normal', 'denoise': 1.0}},
        '14': {'class_type': 'EmptySD3LatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['9',0]}},
        '8': {'class_type': 'VHS_VideoCombine', 'inputs': {'images': ['6',0], 'frame_rate': 16, 'filename_prefix': 'wan_i2v', 'format': 'video/h264-mp4', 'save_output': True, 'loop_count': 0, 'pingpong': False}},
        '9': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'wan_2.1_vae.safetensors'}},
        '10': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'type': 'wan'}}
    }

def flux_t2i(prompt, w=1024, h=1024, steps=20, unet='flux1-dev-Q4_K_S.gguf'):
    return {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': unet}},
        '2': {'class_type': 'EmptyLatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '3': {'class_type': 'DualCLIPLoader', 'inputs': {'clip_name1': 'clip_l.safetensors', 'clip_name2': 't5xxl_fp8_e4m3fn.safetensors', 'type': 'flux'}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['3',0]}},
        '7': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['3',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['4',0], 'negative': ['7',0], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': 1.0, 'sampler_name': 'euler', 'scheduler': 'simple', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['8',0]}},
        '8': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ae.safetensors'}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'flux'}}
    }

def sdxl_t2i(prompt, w=1024, h=1024, steps=4):
    return {
        '1': {'class_type': 'UNETLoader', 'inputs': {'unet_name': 'sdxl_lightning_4step_unet.safetensors', 'weight_dtype': 'fp16'}},
        '2': {'class_type': 'EmptyLatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '3': {'class_type': 'DualCLIPLoader', 'inputs': {'clip_name1': 'clip_l.safetensors', 'clip_name2': 'clip_g.safetensors', 'type': 'sdxl'}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['3',0]}},
        '7': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['3',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['4',0], 'negative': ['7',0], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': 1.0, 'sampler_name': 'euler', 'scheduler': 'sgm_uniform', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['8',0]}},
        '8': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ae.safetensors'}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'sdxl'}}
    }

def sd35_t2i(prompt, w=1024, h=1024, steps=20):
    return {
        '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'sd3.5_medium-Q4_K_S.gguf'}},
        '2': {'class_type': 'EmptySD3LatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '3': {'class_type': 'DualCLIPLoader', 'inputs': {'clip_name1': 'clip_l.safetensors', 'clip_name2': 'clip_g.safetensors', 'type': 'sd3'}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['3',0]}},
        '7': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['3',0]}},
        '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['4',0], 'negative': ['7',0], 'latent_image': ['2',0], 'seed': int(time.time()*1000)%2**32, 'steps': steps, 'cfg': 4.5, 'sampler_name': 'euler', 'scheduler': 'simple', 'denoise': 1.0}},
        '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['8',0]}},
        '8': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ae.safetensors'}},
        '9': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'sd35'}}
    }

# ════════════════════════════════════════════
# ComfyUI Runner
# ════════════════════════════════════════════
async def run_wf(wf):
    unload_audio()
    cid = str(uuid.uuid4())
    async with httpx.AsyncClient(timeout=600) as c:
        r = await c.post(f'{COMFY}/prompt', json={'prompt': wf, 'client_id': cid})
        if r.status_code != 200: return None, f'ComfyUI: {r.text}'
        pid = r.json().get('prompt_id')
        if not pid: return None, f'No prompt_id: {r.text}'
        for _ in range(300):
            await asyncio.sleep(2)
            h = await c.get(f'{COMFY}/history/{pid}')
            d = h.json()
            if pid in d:
                for nid, out in d[pid].get('outputs', {}).items():
                    for k in ['gifs', 'images']:
                        for f in out.get(k, []):
                            fp = os.path.join(OUTPUT, f.get('filename', ''))
                            if os.path.exists(fp): return fp, None
                return None, 'No output file'
        return None, 'Timeout (10 min)'

async def upload_image(image: UploadFile):
    name = f'input_{uuid.uuid4().hex[:8]}.png'
    path = os.path.join(INPUT, name)
    with open(path, 'wb') as f: f.write(await image.read())
    # Upload to ComfyUI
    async with httpx.AsyncClient() as c:
        files = {'image': (name, open(path, 'rb'), 'image/png')}
        await c.post(f'{COMFY}/upload/image', files=files)
    return name

# ════════════════════════════════════════════
# API Endpoints
# ════════════════════════════════════════════
@app.get('/health')
async def health():
    return {
        'status': 'ok',
        'models': {
            'text-to-video': {'ltx': '~2min', 'wan': '~4min'},
            'image-to-video': {'ltx': '~2min', 'wan': '~4min'},
            'text-to-image': {'flux': '~80s', 'flux-schnell': '~15s', 'sdxl': '~30s', 'sd35': '~40s'},
            'speech-to-text': {'whisper': '~5s'},
            'audio-ask': {'qwen2-audio': '~10s'},
            'text-to-speech': {'kokoro': '~2s'}
        }
    }

@app.post('/api/text-to-video')
async def text_to_video(prompt: str = Form(...), model: str = Form('ltx'),
    width: int = Form(768), height: int = Form(512), steps: int = Form(8),
    frames: int = Form(97), api_key: str = Form(...)):
    ck(api_key)
    if model == 'ltx': wf = ltx_t2v(prompt, width, height, steps, frames)
    elif model == 'wan': wf = wan_t2v(prompt, width, height, steps, frames)
    else: raise HTTPException(400, 'model: ltx or wan')
    fp, err = await run_wf(wf)
    if err: raise HTTPException(500, err)
    return FileResponse(fp, media_type='video/mp4')

@app.post('/api/image-to-video')
async def image_to_video(prompt: str = Form(...), image: UploadFile = File(...),
    model: str = Form('ltx'), width: int = Form(768), height: int = Form(512),
    steps: int = Form(8), frames: int = Form(97), api_key: str = Form(...)):
    ck(api_key)
    img_name = await upload_image(image)
    if model == 'ltx': wf = ltx_i2v(prompt, img_name, width, height, steps, frames)
    elif model == 'wan': wf = wan_i2v(prompt, img_name, width, height, steps, frames)
    else: raise HTTPException(400, 'model: ltx or wan')
    fp, err = await run_wf(wf)
    if err: raise HTTPException(500, err)
    return FileResponse(fp, media_type='video/mp4')

@app.post('/api/text-to-image')
async def text_to_image(prompt: str = Form(...), model: str = Form('flux'),
    width: int = Form(1024), height: int = Form(1024), steps: int = Form(20),
    api_key: str = Form(...)):
    ck(api_key)
    if model == 'flux': wf = flux_t2i(prompt, width, height, steps)
    elif model == 'flux-schnell': wf = flux_t2i(prompt, width, height, min(steps,4), 'flux1-schnell-Q4_K_S.gguf')
    elif model == 'sdxl': wf = sdxl_t2i(prompt, width, height, min(steps,4))
    elif model == 'sd35': wf = sd35_t2i(prompt, width, height, steps)
    else: raise HTTPException(400, 'model: flux, flux-schnell, sdxl, sd35')
    fp, err = await run_wf(wf)
    if err: raise HTTPException(500, err)
    return FileResponse(fp, media_type='image/png')

@app.post('/api/speech-to-text')
async def speech_to_text(audio: UploadFile = File(...), api_key: str = Form(...)):
    ck(api_key)
    path = os.path.join(INPUT, f'stt_{uuid.uuid4().hex[:8]}.wav')
    with open(path, 'wb') as f: f.write(await audio.read())
    pipe = load_whisper()
    result = pipe(path, generate_kwargs={'language': 'de', 'task': 'transcribe'})
    return {'text': result['text']}

@app.post('/api/audio-ask')
async def audio_ask(audio: UploadFile = File(...), question: str = Form('Was hoerst du?'),
    api_key: str = Form(...)):
    ck(api_key)
    import librosa
    path = os.path.join(INPUT, f'ask_{uuid.uuid4().hex[:8]}.wav')
    with open(path, 'wb') as f: f.write(await audio.read())
    proc, model = load_qwen()
    conversation = [{'role': 'user', 'content': [
        {'type': 'audio', 'audio_url': path},
        {'type': 'text', 'text': question}
    ]}]
    text = proc.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    audios = [librosa.load(path, sr=proc.feature_extractor.sampling_rate)[0]]
    inputs = proc(text=text, audios=audios, return_tensors='pt', padding=True).to('cuda')
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=512)
    ids = ids[:, inputs.input_ids.size(1):]
    return {'answer': proc.batch_decode(ids, skip_special_tokens=True)[0]}

@app.post('/api/text-to-speech')
async def text_to_speech(text: str = Form(...), voice: str = Form('af_heart'),
    api_key: str = Form(...)):
    ck(api_key)
    import soundfile as sf
    pipe = load_kokoro()
    if not pipe: raise HTTPException(500, 'Kokoro not available')
    samples = []
    for chunk in pipe(text, voice=voice):
        samples.append(chunk[2])
    import numpy as np
    audio = np.concatenate(samples)
    path = os.path.join(OUTPUT, f'tts_{uuid.uuid4().hex[:8]}.wav')
    sf.write(path, audio, 24000)
    return FileResponse(path, media_type='audio/wav')

if __name__ == '__main__':
    uvicorn.run(app, host='0.0.0.0', port=5000)


## 9. API + Tunnel starten

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!pip install fastapi uvicorn python-multipart httpx -q

import subprocess, threading, re, time

api_proc = subprocess.Popen(['python', '/content/api_server.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
time.sleep(3)
print('API server gestartet')

tunnel_url = None
def start_tunnel():
    global tunnel_url
    proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5000'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    for line in proc.stderr:
        m = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
        if m:
            tunnel_url = m.group(1)
            print(f'\n{"="*60}')
            print(f'  CILLIAN AI STUDIO')
            print(f'  URL: {tunnel_url}')
            print(f'  KEY: ironclaw-video-2026')
            print(f'{"="*60}')
            print(f'\n  VIDEO:')
            print(f'    POST /api/text-to-video   model=ltx|wan      ~2-4min')
            print(f'    POST /api/image-to-video  model=ltx|wan      ~2-4min')
            print(f'  BILDER:')
            print(f'    POST /api/text-to-image   model=flux|flux-schnell|sdxl|sd35  ~15-80s')
            print(f'  AUDIO:')
            print(f'    POST /api/speech-to-text  (Whisper)           ~5s')
            print(f'    POST /api/audio-ask       (Qwen2-Audio)       ~10s')
            print(f'    POST /api/text-to-speech  (Kokoro TTS)        ~2s')
            print(f'\n  GET /health  — Alle Modelle + Zeiten')
            break

t = threading.Thread(target=start_tunnel, daemon=True)
t.start()
time.sleep(15)
if not tunnel_url: print('Tunnel noch nicht bereit, warte...')

## 10. Testen

In [ ]:
import requests

if tunnel_url:
    # Health Check
    print('Health:', requests.get(f'{tunnel_url}/health').json())

    # Flux Schnell Test (~15s)
    print('\nTeste Flux Schnell...')
    r = requests.post(f'{tunnel_url}/api/text-to-image', data={
        'prompt': 'a cyberpunk cat with glowing neon eyes, detailed digital art',
        'model': 'flux-schnell', 'width': 768, 'height': 768, 'steps': 4,
        'api_key': 'ironclaw-video-2026'
    }, timeout=300)
    if r.status_code == 200:
        with open('/content/test.png', 'wb') as f: f.write(r.content)
        print(f'Bild: {len(r.content)/1024:.0f} KB')
        from IPython.display import Image, display
        display(Image('/content/test.png', width=512))
    else:
        print(f'Error: {r.status_code} {r.text[:200]}')

In [ ]:
# DEBUG: Check ComfyUI nodes + test simple workflow
import httpx, asyncio, json

async def debug():
    async with httpx.AsyncClient(timeout=30) as c:
        # Check available nodes
        r = await c.get('http://127.0.0.1:8188/object_info')
        nodes = list(r.json().keys())
        check = ['UnetLoaderGGUF','UNETLoader','EmptyLatentImage','KSampler',
                 'VAEDecode','SaveImage','DualCLIPLoader','CLIPLoader',
                 'VHS_VideoCombine','EmptyLTXVLatentVideo','EmptySD3LatentImage',
                 'CLIPVisionLoader','LTXVImgToVideo','unCLIPConditioning']
        print('=== Node Check ===')
        for n in check:
            print(f'  {n}: {"YES" if n in nodes else "NO"}')
        
        # Check models
        print('\n=== Models in unet/ ===')
        import os
        for d in ['unet','vae','clip','loras']:
            p = f'/content/ComfyUI/models/{d}'
            files = os.listdir(p) if os.path.exists(p) else []
            print(f'  {d}/: {files}')

        # Try simple prompt
        print('\n=== Test Simple Workflow ===')
        import time
        wf = {
            '1': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': 'flux1-schnell-Q4_K_S.gguf'}},
            '2': {'class_type': 'EmptyLatentImage', 'inputs': {'width': 256, 'height': 256, 'batch_size': 1}},
            '3': {'class_type': 'DualCLIPLoader', 'inputs': {'clip_name1': 'clip_l.safetensors', 'clip_name2': 't5xxl_fp8_e4m3fn.safetensors', 'type': 'flux'}},
            '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': 'a red circle', 'clip': ['3',0]}},
            '7': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['3',0]}},
            '5': {'class_type': 'KSampler', 'inputs': {'model': ['1',0], 'positive': ['4',0], 'negative': ['7',0], 'latent_image': ['2',0], 'seed': 42, 'steps': 4, 'cfg': 1.0, 'sampler_name': 'euler', 'scheduler': 'simple', 'denoise': 1.0}},
            '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5',0], 'vae': ['8',0]}},
            '8': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'ae.safetensors'}},
            '9': {'class_type': 'SaveImage', 'inputs': {'images': ['6',0], 'filename_prefix': 'debug_test'}}
        }
        r = await c.post('http://127.0.0.1:8188/prompt', json={'prompt': wf})
        print(f'  Submit: {r.status_code}')
        if r.status_code == 200:
            pid = r.json().get('prompt_id')
            print(f'  Prompt ID: {pid}')
            print('  Waiting for result...')
        else:
            print(f'  Error: {r.text[:500]}')

await debug()


## 11. Keep Alive
Diese Zelle laufen lassen damit die Session aktiv bleibt.

In [ ]:
import time
print(f'Cillian AI Studio')
print(f'URL: {tunnel_url}')
print(f'Key: ironclaw-video-2026')
print(f'\n12 Funktionen: T2V, I2V, T2I, STT, Audio-Ask, TTS')
print(f'Laeuft... (Ctrl+M+I zum Stoppen)\n')
i = 0
while True:
    i += 1; time.sleep(300)
    vram = torch.cuda.memory_allocated()/1024**3
    print(f'{i*5}min | VRAM: {vram:.1f}GB | {tunnel_url}')